In [79]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle
from ucimlrepo import fetch_ucirepo 
from sklearn.metrics import roc_auc_score

In [80]:
# fetch dataset 
student_performance = fetch_ucirepo(id=320) 
# this actually loads the dataset related to scores on Portuguese language courses. 

# for the mathematics course (which has 395 instead of 649 instances), you can use the following: 
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student-mat.csv"
# df = pd.read_csv(url, sep=";")
  
# data (as pandas dataframes) 
X = student_performance.data.features 
y = student_performance.data.targets 
  
df_student = pd.concat([X,y],axis=1)

In [81]:
df_student.drop(columns=["G1","G2","school","address"],inplace=True)
df_student.rename(columns={'G3':'target_student'},inplace=True)
df_student["target_student"]=df_student["target_student"].apply(lambda x: 0 if x>=10 else 1) #we predict the fails (1)

# Define feature types
# Numerical (continuous/ordinal numeric): age, failures, absences
numerical_features = ['age', 'failures', 'absences']
# Categorical (binary/ordinal with discrete categories): sex, family size, parents status, education, Likert scales, support flags, etc.
categorical_features = [col for col in df_student.columns if col not in numerical_features and col != 'target_student']
ordinal_features = []
nominal_features = []

feature_order=['sex', 'age', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'traveltime',
       'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities',
       'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime',
       'goout', 'Dalc', 'Walc', 'health', 'absences', 'guardian_father',
       'guardian_mother', 'guardian_other', 'Fjob_at_home', 'Fjob_health',
       'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_course',
       'reason_home', 'reason_other', 'reason_reputation', 'Mjob_at_home',
       'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher','target_student']

In [82]:
df_student.head()

,sex,age,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,target_student
0,F,18,GT3,A,4,4,at_home,teacher,course,mother,...,no,no,4,3,4,1,1,3,4,0
1,F,17,GT3,T,1,1,at_home,other,course,father,...,yes,no,5,3,3,1,1,3,2,0
2,F,15,LE3,T,1,1,at_home,other,other,mother,...,yes,no,4,3,2,2,3,3,6,0
3,F,15,GT3,T,4,2,health,services,home,mother,...,yes,yes,3,2,2,1,1,5,0,0
4,F,16,GT3,T,3,3,other,other,home,father,...,no,no,4,3,2,1,2,5,0,0


In [83]:
# Separate features into already-numeric and categorical-strings
numeric_ordinal = ['traveltime', 'studytime', 'Medu', 'Fedu', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health']
categorical_strings = [col for col in categorical_features if col not in numeric_ordinal]

# Only factorize categorical string features
factorize_mappings = {}
for feature in categorical_strings:
    factorized, uniques = pd.factorize(df_student[feature])
    df_student[feature] = factorized
    # Store mapping for reference
    factorize_mappings[feature] = dict(enumerate(uniques))

# Ensure all values are numeric
df_student = df_student.astype(int)

In [84]:
protected_attributes = ["sex", "age", "health"]

In [85]:
randomstate=123

# Split into train/test
student_train, student_test = train_test_split(df_student, test_size=0.2, random_state=randomstate)

student_train.to_parquet("train_cleaned.parquet")
student_test.to_parquet("test_cleaned.parquet")

# Separate features and target
x_train = student_train.drop("target_student", axis=1)
y_train = student_train["target_student"]
x_test = student_test.drop("target_student", axis=1)
y_test = student_test["target_student"]

In [86]:
x_train.head()

,sex,age,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
243,0,17,0,0,2,1,2,1,0,0,...,0,1,1,3,2,3,1,2,3,0
234,0,17,0,1,2,1,0,2,3,0,...,0,1,0,4,2,5,1,2,5,0
527,1,16,1,1,2,2,3,2,1,0,...,0,0,0,5,1,3,2,2,3,0
177,1,16,0,1,1,3,0,2,0,1,...,1,1,0,5,3,3,1,4,2,2
226,0,16,0,0,2,2,2,1,3,0,...,0,1,0,3,3,4,1,1,4,0


In [87]:
x_train_model = x_train.drop(columns=protected_attributes)
x_test_model = x_test.drop(columns=protected_attributes)

model = RandomForestClassifier(random_state=randomstate, class_weight='balanced')
model.fit(x_train_model, y_train)

target_pred = model.predict(x_test_model)
target_pred_proba = model.predict_proba(x_test_model)[:, 1]  
# evaluate model based on accuracy and AUC metrics
accuracy = accuracy_score(y_test, target_pred)
auc = roc_auc_score(y_test, target_pred_proba)  
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"AUC: {auc * 100:.2f}%")

# Save the model
with open('RF.pkl', 'wb') as f:
    pickle.dump(model, f)

Accuracy: 85.38%
AUC: 82.72%


In [88]:
x_train_model.columns

Index(['famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason',
       'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup',
       'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet',
       'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'absences'],
      dtype='str')

In [89]:
x_train.head()

,sex,age,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
243,0,17,0,0,2,1,2,1,0,0,...,0,1,1,3,2,3,1,2,3,0
234,0,17,0,1,2,1,0,2,3,0,...,0,1,0,4,2,5,1,2,5,0
527,1,16,1,1,2,2,3,2,1,0,...,0,0,0,5,1,3,2,2,3,0
177,1,16,0,1,1,3,0,2,0,1,...,1,1,0,5,3,3,1,4,2,2
226,0,16,0,0,2,2,2,1,3,0,...,0,1,0,3,3,4,1,1,4,0


In [90]:
# Define feature mappings for interpretable output
# Only includes factorized categorical features and numeric ordinal features
FEATURE_MAPPINGS = {
    # Factorized categorical string features
    'sex': {0: 'Female', 1: 'Male'},
    'famsize': {0: '>3', 1: '<=3'},
    'Pstatus': {0: 'Apart', 1: 'Together'},
    'schoolsup': {0: 'yes', 1: 'no'},
    'famsup': {0: 'no', 1: 'yes'},
    'paid': {0: 'no', 1: 'yes'},
    'activities': {0: 'no', 1: 'yes'},
    'nursery': {0: 'yes', 1: 'no'},
    'higher': {0: 'yes', 1: 'no'},
    'internet': {0: 'no', 1: 'yes'},
    'romantic': {0: 'no', 1: 'yes'},
    'Mjob': {0: 'at_home', 1: 'health', 2: 'other', 3: 'services', 4: 'teacher'},
    'Fjob': {0: 'teacher', 1: 'other', 2: 'services', 3: 'health', 4: 'at_home'},
    'reason': {0: 'course', 1: 'other', 2: 'home', 3: 'reputation'},
    'guardian': {0: 'mother', 1: 'father', 2: 'other'},
    
    # Numeric ordinal features (already numeric, 1-5 scales)
    'traveltime': {1: '<15 min', 2: '15-30 min', 3: '30-60 min', 4: '>60 min'},
    'studytime': {1: '<2 hours', 2: '2-5 hours', 3: '5-10 hours', 4: '>10 hours'},
    'Medu': {0: 'none', 1: 'primary', 2: '5-9th grade', 3: 'secondary', 4: 'higher'},
    'Fedu': {0: 'none', 1: 'primary', 2: '5-9th grade', 3: 'secondary', 4: 'higher'},
    'famrel': {1: 'very bad', 2: 'bad', 3: 'fair', 4: 'good', 5: 'very good'},
    'freetime': {1: 'very low', 2: 'low', 3: 'medium', 4: 'high', 5: 'very high'},
    'goout': {1: 'very low', 2: 'low', 3: 'medium', 4: 'high', 5: 'very high'},
    'Dalc': {1: 'very low', 2: 'low', 3: 'medium', 4: 'high', 5: 'very high'},
    'Walc': {1: 'very low', 2: 'low', 3: 'medium', 4: 'high', 5: 'very high'},
    'health': {1: 'very bad', 2: 'bad', 3: 'fair', 4: 'good', 5: 'very good'},
    
    # Numerical features (no mapping)
    'failures': 'numeric (0-3 failures)',
    'absences': 'numeric (0-32 absences)',
}

def map_value(feature_name, value):
    """Map encoded values to readable names"""
    if feature_name in FEATURE_MAPPINGS:
        mapping = FEATURE_MAPPINGS[feature_name]
        if isinstance(mapping, dict):
            if value in mapping:
                return mapping[value]
            # For factorized features, try int conversion
            if isinstance(value, float) and int(value) in mapping:
                return mapping[int(value)]
        else:
            return mapping
    return str(value)



In [91]:
x_train.head()

,sex,age,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
243,0,17,0,0,2,1,2,1,0,0,...,0,1,1,3,2,3,1,2,3,0
234,0,17,0,1,2,1,0,2,3,0,...,0,1,0,4,2,5,1,2,5,0
527,1,16,1,1,2,2,3,2,1,0,...,0,0,0,5,1,3,2,2,3,0
177,1,16,0,1,1,3,0,2,0,1,...,1,1,0,5,3,3,1,4,2,2
226,0,16,0,0,2,2,2,1,3,0,...,0,1,0,3,3,4,1,1,4,0


In [92]:

feature_desc = [
    "The sex of the student",
    "The age of the student in years",
    "Size of the family of the student",
    "Parents cohabitation status",
    "Mother's education",
    "Father's education",
    "Mother's job",
    "Father's job",
    "Reason to choose this school",
    "Student's guardian",
    "Home to school travel time",
    "Weekly study time",
    "Number of past class failures",
    "Student receiving extra educational support",
    "Student receiving family educational support",
    "Student taking extra paid classes within the course subject",
    "Student taking part in extra-curricular activities",
    "Student has attended nursery school",
    "Student wants to get higher education",
    "Student has internet access at home",
    "Student is in a romantic relationship",
    "Quality of family relationships",
    "Free time after school",
    "Going out with friends",
    "Workday alcohol consumption",
    "Weekend alcohol consumption",
    "Current health status",
    "Number of school absences",
]

# Build feature dataframe with distributions for categorical features
feature_data = []
for col in x_test.columns:
    is_cat = col in categorical_features
    row = {
        "feature_name": col,
        "feature_desc": feature_desc[list(x_test.columns).index(col)]
    }
    feature_data.append(row)

feature_desc_df = pd.DataFrame(feature_data)

dataset_description="The dataset contains information about students from two Portugese high schools and in particular their family situation and other habits"
target_description="The target variable represents the final year grade, transformed into whether the student passed (1) or not (0) at the end of the year"
task_description="Predict whether a student will pass"

dataset_info={
 "dataset_description": dataset_description,
 "target_description": target_description,
 "task_description": task_description,
 "feature_description": feature_desc_df
 }


with open('dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

In [93]:
dataset_info


{'dataset_description': 'The dataset contains information about students from two Portugese high schools and in particular their family situation and other habits',
 'target_description': 'The target variable represents the final year grade, transformed into whether the student passed (1) or not (0) at the end of the year',
 'task_description': 'Predict whether a student will pass',
 'feature_description':    feature_name                                       feature_desc
 0           sex                             The sex of the student
 1           age                    The age of the student in years
 2       famsize                  Size of the family of the student
 3       Pstatus                        Parents cohabitation status
 4          Medu                                 Mother's education
 5          Fedu                                 Father's education
 6          Mjob                                       Mother's job
 7          Fjob                               